<a href="https://colab.research.google.com/github/Le2se0hy/FA_ProAn/blob/main/OLSSLRmat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.neighbors import BallTree
from collections import OrderedDict


# ============================================================
# A) 거리 계산: Haversine (입력: rad, 출력: km)  -> lldistkm d1과 동일
# ============================================================
def haversine_km_vec(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * (np.sin(dlon / 2.0) ** 2)
    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    return R * c


# ============================================================
# B) RMSE (MATLAB fitlm 동일): sqrt(SSE/DFE) = sqrt(mse_resid)
# ============================================================
def rmse_fitlm(res):
    return float(np.sqrt(res.mse_resid))


# ============================================================
# C) OLS 적합 (상수항 포함, 결측 drop)
# ============================================================
def fit_model(df, y_col, x_cols):
    X = sm.add_constant(df[x_cols], has_constant="add")
    y = df[y_col]
    return sm.OLS(y, X, missing="drop").fit()


# ============================================================
# D) 유일좌표(sum/count) + 원본 -> 유일좌표 인덱스 매핑
# ============================================================
def make_unique_sum_count(lat, lon, y):
    df_loc = pd.DataFrame({"y": lat, "x": lon, "Y": y})

    # MATLAB unique(rows) 정렬 느낌: sort=True
    g = df_loc.groupby(["y", "x"], sort=True)["Y"].agg(["sum", "count"]).reset_index()
    g["_uix"] = np.arange(len(g), dtype=int)

    lat_uni = g["y"].to_numpy(dtype=float)
    lon_uni = g["x"].to_numpy(dtype=float)
    sumY_uni = g["sum"].to_numpy(dtype=float)
    count_uni = g["count"].to_numpy(dtype=float)

    # 원본 관측치 -> 유일좌표 index
    idx_map = df_loc[["y", "x"]].merge(
        g[["y", "x", "_uix"]],
        on=["y", "x"],
        how="left",
        sort=False
    )["_uix"].to_numpy(dtype=int)

    return lat_uni, lon_uni, sumY_uni, count_uni, idx_map


# ============================================================
# E) WY 계산 (유일좌표 기반, sum/count로 n×n 효과 반영)
#    - 반경: distance_band_km (기본 1km)
#    - 가중치: 1/d
#    - row-standardized
# ============================================================
def compute_WY_unique_counts(lat_uni, lon_uni, sumY_uni, count_uni, distance_band_km=1.0, eps=1e-12):
    lat_r = np.deg2rad(lat_uni.astype(float))
    lon_r = np.deg2rad(lon_uni.astype(float))
    coords = np.column_stack([lat_r, lon_r])

    R = 6371.0
    rad_band = distance_band_km / R
    rad_band_candidate = rad_band * (1.0 + 1e-12)  # 후보 누락 방지

    tree = BallTree(coords, metric="haversine")

    n = len(lat_uni)
    WY_uni = np.zeros(n, dtype=float)

    for i in range(n):
        idx = tree.query_radius(coords[i:i+1], r=rad_band_candidate, return_distance=False)[0]
        idx = idx[idx != i]  # 자기 자신 제거

        if idx.size == 0:
            WY_uni[i] = 0.0
            continue

        d_km = haversine_km_vec(lat_r[i], lon_r[i], lat_r[idx], lon_r[idx])
        mask = (d_km <= distance_band_km + eps) & (d_km > 0)  # 1km 이내 + 거리 0 제외
        idx2 = idx[mask]
        d2 = d_km[mask]

        if idx2.size == 0:
            WY_uni[i] = 0.0
            continue

        invd = 1.0 / d2

        # 유일좌표 기반 n×n 효과 반영:
        # 원래는 동일좌표 관측치들이 각각 이웃으로 들어가므로,
        # 분자: sumY(가격합), 분모: count(개수)로 처리
        num = np.sum(invd * sumY_uni[idx2])
        den = np.sum(invd * count_uni[idx2])
        if den == 0:
            den = 1.0

        WY_uni[i] = num / den

    return WY_uni


# ============================================================
# F) rho 탐색 (RMSE 최소)
#    y_splag = y - rho * WY
# ============================================================
def search_best_rho(df, y_col, x_cols, WY, rho_grid=None):
    if rho_grid is None:
        rho_grid = np.round(np.arange(-0.99, 0.99 + 1e-12, 0.01), 2)

    best_rho = float(rho_grid[0])
    df_tmp = df.copy()
    df_tmp["_WY_"] = WY
    df_tmp["_Y_SPLAG_"] = df_tmp[y_col] - best_rho * df_tmp["_WY_"]

    res_best = fit_model(df_tmp, "_Y_SPLAG_", x_cols)
    best_rmse = rmse_fitlm(res_best)

    for rho in rho_grid[1:]:
        df_tmp["_Y_SPLAG_"] = df_tmp[y_col] - float(rho) * df_tmp["_WY_"]
        res = fit_model(df_tmp, "_Y_SPLAG_", x_cols)
        r = rmse_fitlm(res)
        if r < best_rmse:
            best_rho = float(rho)
            best_rmse = float(r)
            res_best = res

    return best_rho, best_rmse, res_best


# ============================================================
# G) 표 출력용 포맷 (1% 유의 표시 ‡ 포함)
# ============================================================
def mark_1pct(res, var, digits=3):
    if var not in res.params.index:
        return "–"
    coef = res.params[var]
    pval = res.pvalues[var]
    s = f"{coef:.{digits}f}"
    if pval < 0.01:
        s += "‡"
    return s

def fmt_sci_with_mark(res, var, digits=1):
    if var not in res.params.index:
        return "–"
    coef = res.params[var]
    pval = res.pvalues[var]

    if coef == 0:
        s = "0"
    else:
        exp = int(np.floor(np.log10(abs(coef))))
        if exp <= -4:
            mant = coef / (10 ** exp)
            s = f"{mant:.{digits}f} × 10$^{{{exp}}}$"
        else:
            s = f"{coef:.3f}"

    if pval < 0.01:
        s += "‡"
    return s


def build_table(results_dict, col_order, row_map, obs, f_round_to_10=False):
    T = pd.DataFrame(index=list(row_map.keys()), columns=col_order)

    for col in col_order:
        res = results_dict[col]

        # 계수 행
        for rname, vname in row_map.items():
            if vname is None:
                T.loc[rname, col] = ""
                continue

            if rname in ["Units", "Pop. density"]:
                T.loc[rname, col] = fmt_sci_with_mark(res, vname)
            else:
                T.loc[rname, col] = mark_1pct(res, vname)

        # 하단 통계
        fval = res.fvalue
        fp = res.f_pvalue

        if fval is None:
            ftxt = "–"
        else:
            fnum = round(fval, -1) if f_round_to_10 else round(fval)
            ftxt = f"{fnum:,.0f}" + ("‡" if fp < 0.01 else "")

        T.loc["F-statistics", col] = ftxt
        T.loc["RMSE", col] = f"{rmse_fitlm(res):.3f}"
        T.loc["Adjusted $R^2$", col] = f"{res.rsquared_adj:.3f}"

    header = f"Obs.= {obs:,}"
    return header, T


# ============================================================
# H) 메인 실행: WY 만들기 -> rho 찾기 -> Table 4/5 생성
# ============================================================
def run_all(excel_path="0714_busan201819.xlsx", distance_band=1.0,
            print_summary=True, f_round_to_10=False):
    df = pd.read_excel(excel_path)

    # 종속변수/좌표
    Y = df["Price"].astype(float).to_numpy()
    lat = df["y"].astype(float).to_numpy()
    lon = df["x"].astype(float).to_numpy()

    # 1) WY(=IND_spw) 계산 (유일좌표 기반)
    lat_uni, lon_uni, sumY_uni, count_uni, idx_map = make_unique_sum_count(lat, lon, Y)
    WY_uni = compute_WY_unique_counts(lat_uni, lon_uni, sumY_uni, count_uni, distance_band_km=distance_band)
    WY = WY_uni[idx_map]  # 원래 관측치 길이로 복원
    df["IND_spw"] = WY

    # 회귀 변수 세트(논문 Table 4/5의 (1)(2)(3))
    base = ["Area","Floor","Households","Parking","Heating","Year",
            "Dist. Subway","Top Univ.","Sex ratio","Pop. Density",
            "Higher degree","Medium age","Spring","Fall","Winter"]
    x1 = base + ["Dist. Green","Dist. Water"]
    x2 = base + ["Dist. Water","HGVI_50"]
    x3 = base + ["Dist. Green","Dist. Water","HGVI_50"]

    # 2) rho 최적값 탐색(논문 방식: RMSE 최소)
    #    (rho는 x3 전체 변수셋 기준으로 선택)
    rho_best, rmse_net, _ = search_best_rho(df, "Price", x3, df["IND_spw"].to_numpy())

    # 3) Table 4: OLS (Y=Price)
    res_ols = {
        "(1)": fit_model(df, "Price", x1),
        "(2)": fit_model(df, "Price", x2),
        "(3)": fit_model(df, "Price", x3),
    }

    # 4) Table 5: Spatial lag (Y_splag = Y - rho*WY)
    df["Y_splag_net"] = df["Price"] - rho_best * df["IND_spw"]
    res_splag = {
        "(1)": fit_model(df, "Y_splag_net", x1),
        "(2)": fit_model(df, "Y_splag_net", x2),
        "(3)": fit_model(df, "Y_splag_net", x3),
    }

    # Obs (표본 수)
    obs = int(df[["Price"] + x3].dropna().shape[0])

    # 표 행 구성
    row_map = OrderedDict([
        ("Property characteristics", None),
        ("Size", "Area"),
        ("Floor", "Floor"),
        ("Units", "Households"),
        ("Parking", "Parking"),
        ("Heating", "Heating"),
        ("Year", "Year"),

        ("Environmental amenities", None),
        ("Dist. green", "Dist. Green"),
        ("Dist. water", "Dist. Water"),
        ("Green index", "HGVI_50"),

        ("Local built environment", None),
        ("Dist. subway", "Dist. Subway"),
        ("Top univ.", "Top Univ."),

        ("Local demographics", None),
        ("Sex ratio", "Sex ratio"),
        ("Pop. density", "Pop. Density"),
        ("Higher degree", "Higher degree"),
        ("Medium age", "Medium age"),

        ("Seasonality control", None),
        ("Spring", "Spring"),
        ("Fall", "Fall"),
        ("Winter", "Winter"),

        ("F-statistics", "F"),
        ("RMSE", "RMSE"),
        ("Adjusted $R^2$", "AdjR2"),
    ])

    h4, t4 = build_table(res_ols, ["(1)","(2)","(3)"], row_map, obs, f_round_to_10=f_round_to_10)
    h5, t5 = build_table(res_splag, ["(1)","(2)","(3)"], row_map, obs, f_round_to_10=f_round_to_10)

    # 요약 출력(옵션)
    if print_summary:
        print(f"전체 관측치 = {len(df):,} {len(lat_uni):,}")
        print(f"중복 제거 관측치 = {len(lat_uni):,}")
        print("Best rho =", rho_best)
        print("Net RMSE =", rmse_net)
        print("OLS RMSE =", rmse_fitlm(res_ols["(3)"]))

    print("\nTable 4 (OLS):", h4)
    print(t4)
    print("\nTable 5 (Spatial lag):", h5)
    print(t5)

    return {
        "df": df,
        "IND_spw": df["IND_spw"].to_numpy(),
        "rho_net": rho_best,
        "tables": {"table4": t4, "table5": t5},
        "results": {"ols": res_ols, "splag": res_splag},
        "no_all": len(df),
        "no_uni": len(lat_uni),
    }


if __name__ == "__main__":
    # f_round_to_10=True -> F-statistics를 10단위로 반올림해서 표기(논문 표처럼 보이게)
    out = run_all(
        excel_path="0714_busan201819.xlsx",
        distance_band=1.0,
        print_summary=True,
        f_round_to_10=True
    )


전체 관측치 = 52,644 2,404
중복 제거 관측치 = 2,404
Best rho = 0.24
Net RMSE = 0.29195369517026654
OLS RMSE = 0.314679347751799

Table 4 (OLS): Obs.= 52,644
                                       (1)               (2)               (3)
Property characteristics                                                      
Size                                0.012‡            0.012‡            0.012‡
Floor                               0.005‡            0.005‡            0.005‡
Units                     8.4 × 10$^{-5}$‡  8.6 × 10$^{-5}$‡  8.5 × 10$^{-5}$‡
Parking                             0.103‡            0.103‡            0.102‡
Heating                             0.160‡            0.158‡            0.158‡
Year                                0.013‡            0.013‡            0.013‡
Environmental amenities                                                       
Dist. green                        -0.008‡                 –           -0.008‡
Dist. water                        -0.012‡           -0.011‡     